# helpNow — Recomendação de Voluntários
**Global Solution 2026.2 · CitizenTech · Dynamic Programming**

## 1. Cenário
O **helpNow** conecta pessoas cadeirantes e com mobilidade reduzida a voluntários que possam ajudá-las em atividades do dia a dia
(alcançar um objeto numa prateleira, apoio em compras, passar por uma calçada bloqueada, embarcar no transporte).

**Problema computacional:** dada uma solicitação (categoria de ajuda + bairro), encontrar entre todos os voluntários cadastrados
os que estão **aptos** (disponíveis e capacitados para a categoria) e devolvê-los em **ordem de recomendação**:

1. mais perto do bairro do pedido (mesmo bairro → bairro vizinho → mais distante);
2. em caso de empate, mais experiente (mais atendimentos concluídos);
3. em caso de novo empate, menor id (resultado determinístico).

**Origem dos dados:** todos os dados são **simulados** (gerados com semente fixa neste notebook). Os bairros e suas vizinhanças
são da região central de São Paulo (Av. Paulista e arredores); nenhum dado pessoal real é usado.

In [1]:
import random, time, heapq
from collections import deque, defaultdict
from dataclasses import dataclass, field
from functools import lru_cache

random.seed(2026)  # semente fixa: mesmos dados em toda execução

## 2. Dados simulados
### 2.1 Mapa de bairros (grafo)
Cada bairro é um **nó** e cada fronteira entre bairros é uma **aresta**. A distância entre dois bairros é o número de
"saltos" no grafo (0 = mesmo bairro, 1 = vizinho, ...).

In [2]:
VIZINHOS = {
    "Bela Vista":   ["Consolação", "Jardins", "Paraíso", "Liberdade", "República"],
    "Consolação":   ["Bela Vista", "Jardins", "República", "Higienópolis", "Pinheiros"],
    "Jardins":      ["Bela Vista", "Consolação", "Paraíso", "Pinheiros", "Vila Mariana"],
    "Paraíso":      ["Bela Vista", "Jardins", "Liberdade", "Vila Mariana"],
    "Liberdade":    ["Bela Vista", "Paraíso", "República", "Vila Mariana"],
    "República":    ["Bela Vista", "Consolação", "Liberdade", "Higienópolis"],
    "Higienópolis": ["Consolação", "República"],
    "Pinheiros":    ["Consolação", "Jardins"],
    "Vila Mariana": ["Jardins", "Paraíso", "Liberdade"],
}
BAIRROS = list(VIZINHOS)

# o grafo precisa ser não-direcionado: se A é vizinho de B, B é vizinho de A
for a, vs in VIZINHOS.items():
    for b in vs:
        assert a in VIZINHOS[b], f"aresta {a}-{b} sem volta"
print(len(BAIRROS), "bairros,", sum(map(len, VIZINHOS.values())) // 2, "fronteiras")

9 bairros, 17 fronteiras


### 2.2 Categorias, voluntários e solicitações

In [3]:
CATEGORIAS = ["ALCANCAR_OBJETO", "APOIO_COMPRAS", "LOCOMOCAO_CALCADA",
              "EMBARQUE_TRANSPORTE", "ATIVIDADE_COTIDIANA"]

@dataclass
class Voluntario:
    id: int
    bairro: str
    disponivel: bool
    categorias: set = field(default_factory=set)
    atendimentos_concluidos: int = 0

@dataclass
class Solicitacao:
    id: int
    categoria: str
    bairro: str
    descricao: str

def gerar_voluntarios(n):
    return [Voluntario(id=i,
                       bairro=random.choice(BAIRROS),
                       disponivel=random.random() < 0.7,            # ~70% disponíveis
                       categorias=set(random.sample(CATEGORIAS, random.randint(1, 3))),
                       atendimentos_concluidos=random.randint(0, 40))
            for i in range(1, n + 1)]

voluntarios = gerar_voluntarios(30)

solicitacoes = [
    Solicitacao(101, "ALCANCAR_OBJETO", "Bela Vista", "Pegar produto na prateleira alta do mercado"),
    Solicitacao(102, "EMBARQUE_TRANSPORTE", "Pinheiros", "Ajuda para embarcar no ônibus"),
    Solicitacao(103, "LOCOMOCAO_CALCADA", "Vila Mariana", "Calçada bloqueada por obra"),
]

for v in voluntarios[:8]:
    print(v)
print("...")

Voluntario(id=1, bairro='Consolação', disponivel=True, categorias={'ALCANCAR_OBJETO', 'LOCOMOCAO_CALCADA', 'APOIO_COMPRAS'}, atendimentos_concluidos=39)
Voluntario(id=2, bairro='Vila Mariana', disponivel=True, categorias={'EMBARQUE_TRANSPORTE', 'LOCOMOCAO_CALCADA', 'ATIVIDADE_COTIDIANA'}, atendimentos_concluidos=28)
Voluntario(id=3, bairro='Paraíso', disponivel=True, categorias={'ALCANCAR_OBJETO'}, atendimentos_concluidos=18)
Voluntario(id=4, bairro='Consolação', disponivel=True, categorias={'ALCANCAR_OBJETO', 'EMBARQUE_TRANSPORTE', 'LOCOMOCAO_CALCADA'}, atendimentos_concluidos=25)
Voluntario(id=5, bairro='Liberdade', disponivel=True, categorias={'ALCANCAR_OBJETO', 'EMBARQUE_TRANSPORTE'}, atendimentos_concluidos=21)
Voluntario(id=6, bairro='Consolação', disponivel=True, categorias={'EMBARQUE_TRANSPORTE', 'LOCOMOCAO_CALCADA'}, atendimentos_concluidos=9)
Voluntario(id=7, bairro='Liberdade', disponivel=True, categorias={'APOIO_COMPRAS', 'LOCOMOCAO_CALCADA', 'ATIVIDADE_COTIDIANA'}, atendim

## 3. Modelagem — estruturas de dados escolhidas
| Estrutura | Para quê | Por quê |
|---|---|---|
| **Lista de adjacência** (`dict` bairro → vizinhos) | mapa de bairros | grafo esparso; guarda só as fronteiras que existem (O(V+E) de memória) |
| **Índice invertido** (`dict` categoria → `list` de voluntários) | filtro por categoria | em vez de olhar *todos* os voluntários, olha só os que fazem aquela categoria: busca O(1) no `dict` |
| **Busca em largura (BFS) + memoização** (`lru_cache`) | distância entre bairros | a distância de um bairro a todos os outros é calculada **uma única vez** e reaproveitada nos pedidos seguintes |
| **Heap** (`heapq.nsmallest`) | top-k recomendados | quando só os *k* melhores interessam, custa O(m log k) em vez de ordenar tudo |

A **memoização** é a ideia central de programação dinâmica usada aqui: o subproblema "distância do bairro X a todos os outros"
se repete em muitos pedidos, então guardamos a resposta e nunca recalculamos.

In [4]:
def construir_indice(voluntarios):
    """Índice invertido: categoria -> voluntários capacitados. Custo O(n * c)."""
    indice = defaultdict(list)
    for v in voluntarios:
        for cat in v.categorias:
            indice[cat].append(v)
    return indice

@lru_cache(maxsize=None)          # <- MEMOIZAÇÃO: cada bairro de origem é calculado uma vez
def distancias_a_partir_de(origem):
    """BFS no grafo de bairros: devolve {bairro: nº de saltos até a origem}. Custo O(V+E)."""
    dist = {origem: 0}
    fila = deque([origem])
    while fila:
        atual = fila.popleft()
        for viz in VIZINHOS[atual]:
            if viz not in dist:
                dist[viz] = dist[atual] + 1
                fila.append(viz)
    return dist

indice = construir_indice(voluntarios)
print({cat: len(vs) for cat, vs in indice.items()})
print(distancias_a_partir_de("Pinheiros"))

{'ALCANCAR_OBJETO': 15, 'LOCOMOCAO_CALCADA': 13, 'APOIO_COMPRAS': 9, 'EMBARQUE_TRANSPORTE': 14, 'ATIVIDADE_COTIDIANA': 14}
{'Pinheiros': 0, 'Consolação': 1, 'Jardins': 1, 'Bela Vista': 2, 'República': 2, 'Higienópolis': 2, 'Paraíso': 2, 'Vila Mariana': 2, 'Liberdade': 3}


## 4. Algoritmo de recomendação
1. **Busca** no índice invertido os voluntários da categoria pedida (O(1)).
2. **Filtra** os indisponíveis (O(m), m = voluntários da categoria).
3. **Calcula a prioridade** de cada candidato com a tabela de distâncias memoizada:
   `chave = (distância_do_bairro, -atendimentos_concluidos, id)` — tuplas em Python são comparadas campo a campo,
   então essa chave já implementa as 3 regras de desempate.
4. **Ordena** pela chave (O(m log m)) ou, se só os *k* primeiros importam, usa **heap** (O(m log k)).

In [5]:
def recomendar(solicitacao, indice, k=None):
    """Devolve os voluntários aptos para a solicitação, em ordem de recomendação."""
    candidatos = indice.get(solicitacao.categoria, [])            # 1. busca O(1)
    aptos = [v for v in candidatos if v.disponivel]                # 2. filtro O(m)
    dist = distancias_a_partir_de(solicitacao.bairro)             # 3. tabela memoizada
    chave = lambda v: (dist.get(v.bairro, float("inf")),           #    regra 1: proximidade
                       -v.atendimentos_concluidos,                 #    regra 2: experiência
                       v.id)                                       #    regra 3: desempate
    if k is None:
        return sorted(aptos, key=chave)                            # 4a. O(m log m)
    return heapq.nsmallest(k, aptos, key=chave)                    # 4b. O(m log k)

def mostrar(sol, recomendados):
    dist = distancias_a_partir_de(sol.bairro)
    print(f"Solicitação {sol.id} | {sol.categoria} | {sol.bairro} — {sol.descricao}")
    if not recomendados:
        print("   Nenhum voluntário apto. Encaminhar para ponto comunitário de apoio.")
    for pos, v in enumerate(recomendados, 1):
        rel = ["mesmo bairro", "bairro vizinho"][dist[v.bairro]] if dist[v.bairro] < 2 else f"{dist[v.bairro]} bairros"
        print(f"   {pos}º  voluntário #{v.id:<3} {v.bairro:<13} ({rel:<14}) {v.atendimentos_concluidos:>2} atendimentos")
    print()

## 5. Execução — entradas e saídas

In [6]:
for sol in solicitacoes:
    mostrar(sol, recomendar(sol, indice, k=5))

Solicitação 101 | ALCANCAR_OBJETO | Bela Vista — Pegar produto na prateleira alta do mercado
   1º  voluntário #1   Consolação    (bairro vizinho) 39 atendimentos
   2º  voluntário #8   Consolação    (bairro vizinho) 27 atendimentos
   3º  voluntário #4   Consolação    (bairro vizinho) 25 atendimentos
   4º  voluntário #5   Liberdade     (bairro vizinho) 21 atendimentos
   5º  voluntário #3   Paraíso       (bairro vizinho) 18 atendimentos

Solicitação 102 | EMBARQUE_TRANSPORTE | Pinheiros — Ajuda para embarcar no ônibus
   1º  voluntário #13  Pinheiros     (mesmo bairro  ) 25 atendimentos
   2º  voluntário #17  Pinheiros     (mesmo bairro  ) 18 atendimentos
   3º  voluntário #8   Consolação    (bairro vizinho) 27 atendimentos
   4º  voluntário #4   Consolação    (bairro vizinho) 25 atendimentos
   5º  voluntário #6   Consolação    (bairro vizinho)  9 atendimentos

Solicitação 103 | LOCOMOCAO_CALCADA | Vila Mariana — Calçada bloqueada por obra
   1º  voluntário #2   Vila Mariana  (mesmo

## 6. Testes
Verificamos que a saída **respeita todas as regras**, comparando com uma solução por força bruta (sem índice, sem memoização).

In [7]:
def recomendar_forca_bruta(sol, voluntarios):
    """Referência ingênua: percorre todos os voluntários e roda uma BFS por voluntário."""
    def bfs(a, b):
        dist, fila = {a: 0}, deque([a])
        while fila:
            x = fila.popleft()
            for y in VIZINHOS[x]:
                if y not in dist:
                    dist[y] = dist[x] + 1; fila.append(y)
        return dist[b]
    aptos = [v for v in voluntarios if v.disponivel and sol.categoria in v.categorias]
    return sorted(aptos, key=lambda v: (bfs(sol.bairro, v.bairro), -v.atendimentos_concluidos, v.id))

for sol in solicitacoes:
    rec = recomendar(sol, indice)
    # 1) todos aptos: disponíveis e da categoria certa
    assert all(v.disponivel and sol.categoria in v.categorias for v in rec)
    # 2) ninguém apto ficou de fora
    assert len(rec) == sum(v.disponivel and sol.categoria in v.categorias for v in voluntarios)
    # 3) mesma ordem da referência por força bruta
    assert [v.id for v in rec] == [v.id for v in recomendar_forca_bruta(sol, voluntarios)]
    # 4) top-k por heap = primeiros k da ordenação completa
    assert [v.id for v in recomendar(sol, indice, k=3)] == [v.id for v in rec[:3]]

# 5) caso de borda: categoria sem nenhum voluntário
assert recomendar(Solicitacao(999, "CATEGORIA_INEXISTENTE", "Bela Vista", "-"), indice) == []
# 6) caso de borda: ninguém disponível
assert recomendar(solicitacoes[0], construir_indice([Voluntario(1, "Bela Vista", False, {"ALCANCAR_OBJETO"})])) == []
print("Todos os testes passaram ✔")

Todos os testes passaram ✔


## 7. Eficiência
### 7.1 Complexidade
Sejam **n** voluntários, **c** categorias por voluntário (no máx. 3), **m** voluntários da categoria pedida (m ≤ n),
**V** bairros, **E** fronteiras e **k** recomendados desejados.

| Etapa | Tempo | Espaço |
|---|---|---|
| Construir índice invertido (uma vez) | O(n·c) | O(n·c) |
| BFS de um bairro (uma vez por bairro, graças à memoização) | O(V+E) | O(V) |
| Buscar categoria no índice | O(1) | — |
| Filtrar disponíveis | O(m) | O(m) |
| Ordenar todos / top-k com heap | O(m log m) / O(m log k) | O(m) / O(k) |
| **Por solicitação (depois do pré-processamento)** | **O(m log k)** | **O(m)** |
| Força bruta (referência) | O(n·(V+E) + n log n) | O(n) |

### 7.2 Medição prática
Comparamos o tempo das duas versões com bases cada vez maiores.

In [8]:
import matplotlib.pyplot as plt

tamanhos = [1_000, 5_000, 10_000, 20_000, 40_000]
t_otim, t_bruta = [], []
sol = solicitacoes[0]
for n in tamanhos:
    random.seed(n)
    base = gerar_voluntarios(n)
    idx = construir_indice(base)
    distancias_a_partir_de.cache_clear()

    t0 = time.perf_counter()
    for _ in range(20): recomendar(sol, idx, k=5)
    t_otim.append((time.perf_counter() - t0) / 20 * 1000)

    t0 = time.perf_counter()
    recomendar_forca_bruta(sol, base)
    t_bruta.append((time.perf_counter() - t0) * 1000)

for n, a, b in zip(tamanhos, t_otim, t_bruta):
    print(f"n={n:>6}: otimizado {a:7.2f} ms | força bruta {b:8.2f} ms | {b/a:5.1f}x mais rápido")

plt.figure(figsize=(7, 4))
plt.plot(tamanhos, t_bruta, "o-", label="Força bruta (BFS por voluntário + sort)")
plt.plot(tamanhos, t_otim, "o-", label="helpNow (índice + memoização + heap)")
plt.xlabel("Voluntários cadastrados"); plt.ylabel("Tempo por solicitação (ms)")
plt.title("Tempo de recomendação por solicitação"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout()
plt.savefig("grafico_desempenho.png", dpi=110)   # salvo como arquivo separado: o notebook fica leve
plt.close()
print("Gráfico salvo em grafico_desempenho.png")

n=  1000: otimizado    0.13 ms | força bruta     0.96 ms |   7.2x mais rápido
n=  5000: otimizado    0.55 ms | força bruta     5.09 ms |   9.2x mais rápido
n= 10000: otimizado    0.69 ms | força bruta     7.99 ms |  11.6x mais rápido
n= 20000: otimizado    1.14 ms | força bruta    15.77 ms |  13.9x mais rápido
n= 40000: otimizado    2.45 ms | força bruta    34.89 ms |  14.2x mais rápido


Gráfico salvo em grafico_desempenho.png


![Tempo de recomendação por solicitação](grafico_desempenho.png)

## 8. Como o algoritmo beneficia o helpNow
- **Resposta rápida:** o pedido chega primeiro a quem está mais perto, o que reduz o tempo de espera de quem está parado diante de uma barreira.
- **Justo e previsível:** as regras de prioridade são explícitas e determinísticas; o mesmo pedido sempre gera a mesma ordem.
- **Escala:** com índice + memoização, o custo por pedido depende só dos voluntários daquela categoria, não da base inteira.
- **Fallback:** quando ninguém está apto, o sistema sabe na hora e pode encaminhar o pedido ao ponto comunitário de apoio.

**Limitações (cenário simulado):** a distância é medida em bairros, não em metros; disponibilidade é um valor fixo.
Evoluções naturais: geolocalização real, tempo de deslocamento por rota acessível e avaliação dos voluntários no critério de prioridade.